# 05 — Silver Transactions

**Input:** `Capstone_team01.bronze.card_transactions`  
**Outputs:** `Capstone_team01.silver.card_transactions` and `quarantine_card_transactions`

Run `08_silver_master_data` first so reference checks can use customer, merchant, and card Silver tables.

Rules:
- `transaction_id` is required and unique.
- `transaction_ts` must parse as a timestamp when present.
- `amount` must parse as a decimal when present.
- Customer, merchant, and card references must exist in Silver master tables.
- Duplicate transaction IDs are quarantined in full.
- No arbitrary amount threshold or undocumented enum list is assumed.


In [ ]:
from pyspark.sql import functions
from pyspark.sql.window import Window

spark.sql("USE CATALOG Capstone_team01")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

df = spark.table("Capstone_team01.bronze.card_transactions")

# Preserve original timestamp and amount for invalid-cast detection.
df = (df
      .withColumn("_raw_transaction_ts", functions.col("transaction_ts"))
      .withColumn("_raw_amount", functions.col("amount")))

df = (df
      .withColumn("transaction_ts", functions.to_timestamp("transaction_ts"))
      .withColumn("amount", functions.col("amount").cast("decimal(18,2)")))

duplicate_window = Window.partitionBy("transaction_id")
df = df.withColumn("_key_count", functions.count("*").over(duplicate_window))

reasons = [
    functions.when(
        functions.col("transaction_id").isNull() |
        (functions.trim(functions.col("transaction_id")) == ""),
        functions.lit("MISSING_TRANSACTION_ID")
    ),
    functions.when(functions.col("_key_count") > 1,
                   functions.lit("DUPLICATE_TRANSACTION_ID")),
    functions.when(functions.col("_raw_transaction_ts").isNotNull() &
                   functions.col("transaction_ts").isNull(),
                   functions.lit("INVALID_TRANSACTION_TS")),
    functions.when(functions.col("_raw_amount").isNotNull() &
                   functions.col("amount").isNull(),
                   functions.lit("INVALID_AMOUNT")),
    functions.when(functions.col("_rescued_data").isNotNull(),
                   functions.lit("AUTOLOADER_RESCUED_DATA"))
]

df = df.withColumn("_dq_reason", functions.concat_ws(";", *reasons))

# Reference checks against cleaned master data.
customers = spark.table("Capstone_team01.silver.customer_master").select("customer_id").distinct()
merchants = spark.table("Capstone_team01.silver.merchant_master").select("merchant_id").distinct()
cards = spark.table("Capstone_team01.silver.card_master").select("card_id").distinct()

df = (df
      .join(customers.withColumn("_customer_found", functions.lit(True)), "customer_id", "left")
      .join(merchants.withColumn("_merchant_found", functions.lit(True)), "merchant_id", "left")
      .join(cards.withColumn("_card_found", functions.lit(True)), "card_id", "left"))

df = df.withColumn(
    "_dq_reason",
    functions.concat_ws(
        ";",
        functions.col("_dq_reason"),
        functions.when(functions.col("_customer_found").isNull(), "UNKNOWN_CUSTOMER_ID"),
        functions.when(functions.col("_merchant_found").isNull(), "UNKNOWN_MERCHANT_ID"),
        functions.when(functions.col("_card_found").isNull(), "UNKNOWN_CARD_ID")
    )
)

df = df.withColumn("_quarantined_at", functions.current_timestamp())

quarantine_df = df.filter(functions.col("_dq_reason") != "")
clean_df = df.filter(functions.col("_dq_reason") == "")

clean_df = clean_df.drop(
    "_raw_transaction_ts", "_raw_amount", "_key_count", "_dq_reason",
    "_quarantined_at", "_customer_found", "_merchant_found", "_card_found"
)

(
    clean_df.write.format("delta").mode("overwrite")
    .saveAsTable("Capstone_team01.silver.card_transactions")
)
(
    quarantine_df.write.format("delta").mode("overwrite")
    .saveAsTable("Capstone_team01.silver.quarantine_card_transactions")
)

print("Clean transactions:", clean_df.count())
print("Quarantined transactions:", quarantine_df.count())
display(clean_df.limit(10))
